In [2]:
import warnings
warnings.filterwarnings("ignore")

import time
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

X_train, X_test, y_train, y_test = joblib.load("../data/splits.joblib")
results = []

def evaluate(name, model):
    start = time.time()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "AUPRC": average_precision_score(y_test, proba),
        "Seconds": round(time.time() - start),
    })

In [3]:
evaluate("Random Forest", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))

In [4]:
evaluate("XGBoost", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                  n_jobs=-1, random_state=42))

In [5]:
evaluate("LightGBM", LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                    n_jobs=-1, random_state=42, verbose=-1))

In [6]:
table = pd.DataFrame(results).round(3).sort_values("AUPRC", ascending=False)
print(table)

           Model  Precision  Recall     F1  AUPRC  Seconds
1        XGBoost      0.909   0.816  0.860  0.877        1
0  Random Forest      0.941   0.816  0.874  0.873       25
2       LightGBM      0.907   0.796  0.848  0.861        2


In [7]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_child_weight": [1, 3, 5],
}

search = RandomizedSearchCV(
    XGBClassifier(n_jobs=-1, random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    scoring="average_precision",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best settings:", search.best_params_)
print("Best CV AUPRC:", round(search.best_score_, 3))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best settings: {'subsample': 0.7, 'n_estimators': 500, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.85}
Best CV AUPRC: 0.854


In [8]:
best = search.best_estimator_
pred = best.predict(X_test)
proba = best.predict_proba(X_test)[:, 1]

print("Precision:", round(precision_score(y_test, pred), 3))
print("Recall   :", round(recall_score(y_test, pred), 3))
print("F1       :", round(f1_score(y_test, pred), 3))
print("AUPRC    :", round(average_precision_score(y_test, proba), 3))

Precision: 0.93
Recall   : 0.816
F1       : 0.87
AUPRC    : 0.879


In [9]:
joblib.dump(best, "../models/xgb_model.joblib")

['../models/xgb_model.joblib']